# Tire Quality Classification — Image Processing Pipeline

**Dataset:** [Tyre Quality Classification – Kaggle](https://www.kaggle.com/datasets/warcoder/tyre-quality-classification)  
**Classes:** `good` (0) | `defective` (1)  
**Theme:** Cars and Motorcycles

### Pipeline overview
1. Dataset exploration  
2. Pre-processing — spatial domain (manual Gaussian blur + histogram equalisation)  
3. Frequency domain — manual block DCT  
4. Descriptive analysis — entropy, mean, std, skewness, kurtosis  
5. Classical pipeline — manual Canny, Otsu segmentation, morphology  
6. Feature assembly and exploration  
7. Machine learning — SVM, KNN, Logistic Regression (with hyperparameter study)  
8. Results and interpretation

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

# Project modules
from src.preprocessing import preprocess, load_image, to_grayscale, resize_image, gaussian_blur, histogram_equalize
from src.frequency import block_dct, freq_energy_ratio
from src.descriptors import entropy, image_statistics, pixel_histogram
from src.edges import canny, edge_density, segment_tire, defect_area_ratio, otsu_threshold, sobel_gradient
from src.morphology import closing, rect_kernel, connected_regions_count, erode, dilate, opening
from src.features import build_feature_vector, load_dataset, FEATURE_NAMES

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_DIR   = Path('../data/tyre_dataset')
FIGURES    = Path('../results/figures')
METRICS    = Path('../results/metrics')
FIGURES.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Setup complete.')

---
## Section 1 — Dataset Exploration

In [ ]:
good_dir      = DATA_DIR / 'good'
defective_dir = DATA_DIR / 'defective'

EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

good_paths      = sorted([p for p in good_dir.iterdir()      if p.suffix.lower() in EXTS])
defective_paths = sorted([p for p in defective_dir.iterdir() if p.suffix.lower() in EXTS])

print(f"Good tires     : {len(good_paths):>5}")
print(f"Defective tires: {len(defective_paths):>5}")
print(f"Total          : {len(good_paths) + len(defective_paths):>5}")

In [ ]:
# Class balance bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Good', 'Defective'], [len(good_paths), len(defective_paths)],
            color=['#4CAF50', '#F44336'], edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Number of images')
for i, v in enumerate([len(good_paths), len(defective_paths)]):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Image size distribution
sample_paths = random.sample(good_paths + defective_paths, min(200, len(good_paths) + len(defective_paths)))
heights, widths = [], []
for p in sample_paths:
    img = cv2.imread(str(p))
    if img is not None:
        heights.append(img.shape[0])
        widths.append(img.shape[1])

axes[1].scatter(widths, heights, alpha=0.5, s=10)
axes[1].set_title('Image Dimensions (sample of 200)')
axes[1].set_xlabel('Width (px)')
axes[1].set_ylabel('Height (px)')

plt.tight_layout()
plt.savefig(FIGURES / '01_class_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Sample grid: 3 good + 3 defective
sample_good = random.sample(good_paths, 3)
sample_def  = random.sample(defective_paths, 3)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for col, p in enumerate(sample_good):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'Good #{col+1}', color='green')
    axes[0, col].axis('off')

for col, p in enumerate(sample_def):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    axes[1, col].imshow(img)
    axes[1, col].set_title(f'Defective #{col+1}', color='red')
    axes[1, col].axis('off')

plt.suptitle('Dataset Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '01_sample_grid.png', bbox_inches='tight')
plt.show()

---
## Section 2 — Pre-processing (Spatial Domain)

**Implemented manually:** `to_grayscale` (luminosity weights), `gaussian_blur` (2-D convolution with Gaussian kernel), `histogram_equalize` (CDF mapping).  
OpenCV is used **only** for `imread` / `resize`.

In [ ]:
demo_paths = [sample_good[0], sample_def[0]]

stage_titles = ['Original (BGR→RGB)', 'Grayscale', 'Resized 256×256',
                'Gaussian Blur', 'Histogram Equalised']

fig, axes = plt.subplots(len(demo_paths), 5, figsize=(18, 6))
row_labels = ['Good', 'Defective']

for row, path in enumerate(demo_paths):
    original = cv2.imread(str(path))
    gray     = to_grayscale(original)
    resized  = resize_image(gray, (256, 256))
    blurred  = gaussian_blur(resized)
    equalized = histogram_equalize(blurred)

    stages = [
        cv2.cvtColor(original, cv2.COLOR_BGR2RGB),
        gray, resized, blurred, equalized
    ]
    cmaps = [None, 'gray', 'gray', 'gray', 'gray']

    for col, (stage, title, cmap) in enumerate(zip(stages, stage_titles, cmaps)):
        axes[row, col].imshow(stage, cmap=cmap)
        if row == 0:
            axes[row, col].set_title(title)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(row_labels[row], fontsize=12, fontweight='bold', rotation=0, labelpad=60)

plt.suptitle('Pre-processing Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '02_preprocessing.png', bbox_inches='tight')
plt.show()

---
## Section 3 — Frequency Domain (Manual Block DCT)

The image is split into 8×8 blocks. The DCT-II is applied manually (no `numpy.fft` or `scipy.fft`).  
**Feature:** `freq_energy_ratio` — fraction of block energy in the top-left 2×2 DC coefficients.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
row_labels = ['Good', 'Defective']

for row, path in enumerate(demo_paths):
    stages = preprocess(str(path))
    eq = stages['equalized']
    dct_img = block_dct(eq)
    ratio   = freq_energy_ratio(dct_img)

    # Log-magnitude for visualisation
    dct_vis = np.log1p(np.abs(dct_img))

    axes[row, 0].imshow(eq, cmap='gray')
    axes[row, 0].set_title('Equalised Input' if row == 0 else '')
    axes[row, 0].axis('off')

    im = axes[row, 1].imshow(dct_vis, cmap='hot')
    axes[row, 1].set_title('Block DCT (log|coeff|)' if row == 0 else '')
    axes[row, 1].axis('off')
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046)

    # Energy distribution: sum of squared coefficients per block row
    block_energies = []
    for r in range(0, eq.shape[0] - 7, 8):
        row_energy = (dct_img[r:r+8, :] ** 2).mean()
        block_energies.append(row_energy)
    axes[row, 2].plot(block_energies)
    axes[row, 2].set_title(f'Block Row Energy  (ratio={ratio:.3f})' if row == 0 else f'freq_energy_ratio={ratio:.3f}')
    axes[row, 2].set_xlabel('Block row')
    axes[row, 2].set_ylabel('Mean energy')

    axes[row, 0].set_ylabel(row_labels[row], fontsize=12, fontweight='bold', rotation=0, labelpad=60)

plt.suptitle('Frequency Domain Analysis — Manual Block DCT', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '03_dct.png', bbox_inches='tight')
plt.show()

---
## Section 4 — Descriptive Analysis

**Features computed per image:** entropy, mean, std, skewness, kurtosis  
We show distributions separated by class to validate discriminative power.

In [ ]:
# Compute descriptors for a representative sample (100 per class)
n_sample = min(100, len(good_paths), len(defective_paths))
sample_g = random.sample(good_paths, n_sample)
sample_d = random.sample(defective_paths, n_sample)

def compute_descriptors(paths):
    rows = []
    for p in paths:
        stages = preprocess(str(p))
        eq = stages['equalized']
        stats = image_statistics(eq)
        stats['entropy'] = entropy(eq)
        rows.append(stats)
    return pd.DataFrame(rows)

df_good = compute_descriptors(sample_g)
df_def  = compute_descriptors(sample_d)
print("Good tires — mean statistics:")
print(df_good.mean().round(3))
print("\nDefective tires — mean statistics:")
print(df_def.mean().round(3))

In [ ]:
features_desc = ['entropy', 'mean', 'std', 'skewness', 'kurtosis']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, feat in zip(axes, features_desc):
    ax.hist(df_good[feat],  bins=20, alpha=0.6, label='Good',      color='#4CAF50', edgecolor='white')
    ax.hist(df_def[feat],   bins=20, alpha=0.6, label='Defective', color='#F44336', edgecolor='white')
    ax.set_title(feat.capitalize())
    ax.set_xlabel('Value')
    ax.legend(fontsize=8)

plt.suptitle('Descriptor Distributions by Class (sample n=100 each)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '04_descriptors.png', bbox_inches='tight')
plt.show()

---
## Section 5 — Classical Pipeline: Canny Edges, Otsu Segmentation, Morphology

All implemented manually:
- **Canny**: Sobel gradients → Non-maximum suppression → Hysteresis thresholding  
- **Otsu**: Minimise intra-class variance analytically  
- **Morphology**: Erosion / dilation / opening / **closing** (connects crack fragments)

In [ ]:
pipeline_paths = [sample_good[1], sample_good[2], sample_def[1], sample_def[2]]
pipeline_labels = ['Good #2', 'Good #3', 'Defective #2', 'Defective #3']
stage_names = ['Equalised', 'Canny Edges', 'Otsu Mask', 'After Closing']

fig, axes = plt.subplots(4, 4, figsize=(16, 14))
kernel = rect_kernel(3)

for row, (path, label) in enumerate(zip(pipeline_paths, pipeline_labels)):
    stages = preprocess(str(path))
    eq = stages['equalized']
    edges = canny(eq)
    mask  = segment_tire(eq)
    mask_closed = closing(mask, kernel)

    imgs = [eq, edges, mask, mask_closed]
    for col, (img, title) in enumerate(zip(imgs, stage_names)):
        axes[row, col].imshow(img, cmap='gray')
        if row == 0:
            axes[row, col].set_title(title, fontsize=11)
        axes[row, col].axis('off')

    axes[row, 0].set_ylabel(label, fontsize=10, fontweight='bold', rotation=0, labelpad=70)

plt.suptitle('Classical Pipeline: Edges → Segmentation → Morphology', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '05_classical_pipeline.png', bbox_inches='tight')
plt.show()

In [ ]:
# Show morphological sequence: original mask → erode → dilate → opening → closing
demo_mask_path = sample_def[0]
stages_d = preprocess(str(demo_mask_path))
eq_d = stages_d['equalized']
mask_d = segment_tire(eq_d)

morph_results = {
    'Otsu Mask': mask_d,
    'Erosion': erode(mask_d, kernel),
    'Dilation': dilate(mask_d, kernel),
    'Opening': opening(mask_d, kernel),
    'Closing': closing(mask_d, kernel),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (title, img) in zip(axes, morph_results.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.suptitle('Morphological Operations — Defective Tire', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '05_morphology.png', bbox_inches='tight')
plt.show()

---
## Section 6 — Feature Assembly and Exploration

Extract the 9-feature vector from all images in the dataset.  
**Note:** This step can take several minutes due to the manual implementations.

> Run time can be reduced by setting `LIMIT_SAMPLE` to a smaller integer for testing.

In [ ]:
CSV_PATH = METRICS / 'features.csv'

if CSV_PATH.exists():
    print(f"Loading cached features from {CSV_PATH}")
    df_feat = pd.read_csv(CSV_PATH)
    X = df_feat[FEATURE_NAMES].values
    y = df_feat['label'].values
    paths = df_feat['path'].tolist()
else:
    print("Computing features for all images (this may take a while)...")
    X, y, paths = load_dataset(DATA_DIR, save_csv=CSV_PATH)

print(f"Feature matrix shape: {X.shape}")
print(f"Label distribution  : good={( y==0).sum()}, defective={(y==1).sum()}")

In [ ]:
# Feature correlation heatmap
df_features = pd.DataFrame(X, columns=FEATURE_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

corr = df_features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[0], linewidths=0.5)
axes[0].set_title('Feature Correlation Matrix')

# PCA 2D scatter
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X)
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

colors = np.where(y == 0, '#4CAF50', '#F44336')
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.5, s=15)
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor='#4CAF50', label='Good'),
                  Patch(facecolor='#F44336', label='Defective')]
axes[1].legend(handles=legend_handles)
axes[1].set_title(f'PCA 2D Projection (var explained: {pca.explained_variance_ratio_.sum()*100:.1f}%)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.savefig(FIGURES / '06_feature_exploration.png', bbox_inches='tight')
plt.show()

---
## Section 7 — Machine Learning

Pipeline: StandardScaler → Classifier  
- **SVM (RBF):** GridSearchCV for C and gamma  
- **KNN:** accuracy vs k curve  
- **Logistic Regression:** baseline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}  Test: {X_test_s.shape}")
print(f"Train — good: {(y_train==0).sum()}, defective: {(y_train==1).sum()}")
print(f"Test  — good: {(y_test==0).sum()},  defective: {(y_test==1).sum()}")

In [ ]:
# ── SVM with GridSearchCV ────────────────────────────────────────────────────
param_grid_svm = {
    'C':     [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01],
    'kernel': ['rbf'],
}
svm_gs = GridSearchCV(SVC(random_state=SEED), param_grid_svm, cv=5,
                      scoring='f1', n_jobs=-1, verbose=1)
svm_gs.fit(X_train_s, y_train)

best_svm = svm_gs.best_estimator_
y_pred_svm = best_svm.predict(X_test_s)

print(f"\nBest SVM params: {svm_gs.best_params_}")
print(classification_report(y_test, y_pred_svm, target_names=['good', 'defective']))

In [ ]:
# ── KNN — hyperparameter study ───────────────────────────────────────────────
k_values = list(range(1, 16, 2))
knn_train_accs, knn_test_accs, knn_f1s = [], [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    knn_train_accs.append(accuracy_score(y_train, knn.predict(X_train_s)))
    y_pred_k = knn.predict(X_test_s)
    knn_test_accs.append(accuracy_score(y_test, y_pred_k))
    knn_f1s.append(f1_score(y_test, y_pred_k))

best_k = k_values[np.argmax(knn_test_accs)]
print(f"Best k: {best_k} — Test accuracy: {max(knn_test_accs):.4f}")

best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train_s, y_train)
y_pred_knn = best_knn.predict(X_test_s)
print(classification_report(y_test, y_pred_knn, target_names=['good', 'defective']))

In [ ]:
# ── Logistic Regression baseline ────────────────────────────────────────────
lr = LogisticRegression(max_iter=500, random_state=SEED)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
print(classification_report(y_test, y_pred_lr, target_names=['good', 'defective']))

In [ ]:
# ── Comparison visualisation ─────────────────────────────────────────────────
models = {
    'SVM (RBF)':   (y_pred_svm, '#1565C0'),
    f'KNN (k={best_k})': (y_pred_knn, '#6A1B9A'),
    'Logistic Reg': (y_pred_lr,  '#2E7D32'),
}

results = {}
for name, (y_pred, color) in models.items():
    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1-score':  f1_score(y_test, y_pred, zero_division=0),
    }

df_results = pd.DataFrame(results).T
df_results.to_csv(METRICS / 'model_comparison.csv')
print(df_results.round(4))

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig)

# ── KNN k-curve ──────────────────────────────────────────────────────────────
ax_k = fig.add_subplot(gs[0, 0])
ax_k.plot(k_values, knn_train_accs, 'o--', label='Train accuracy', color='navy')
ax_k.plot(k_values, knn_test_accs,  's-',  label='Test accuracy',  color='darkorange')
ax_k.axvline(x=best_k, linestyle=':', color='gray', label=f'Best k={best_k}')
ax_k.set_title('KNN: Accuracy vs k')
ax_k.set_xlabel('k')
ax_k.set_ylabel('Accuracy')
ax_k.legend()
ax_k.set_xticks(k_values)

# ── Confusion matrices ────────────────────────────────────────────────────────
positions = [(0, 1), (0, 2), (1, 0)]
for pos, (name, (y_pred, color)) in zip(positions, models.items()):
    ax = fig.add_subplot(gs[pos])
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Good', 'Defective'],
                yticklabels=['Good', 'Defective'])
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    ax.set_title(f'{name}\nAcc={acc:.3f}  F1={f1:.3f}')
    ax.set_ylabel('True label')
    ax.set_xlabel('Predicted label')

# ── Bar chart comparison ──────────────────────────────────────────────────────
ax_bar = fig.add_subplot(gs[1, 1:])
x = np.arange(len(df_results))
width = 0.2
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-score']
bar_colors = ['#1565C0', '#6A1B9A', '#2E7D32', '#F57F17']
for i, (metric, bar_color) in enumerate(zip(metrics_list, bar_colors)):
    ax_bar.bar(x + i * width, df_results[metric], width=width,
               label=metric, color=bar_color, alpha=0.85)
ax_bar.set_xticks(x + 1.5 * width)
ax_bar.set_xticklabels(df_results.index)
ax_bar.set_ylim(0, 1.1)
ax_bar.set_ylabel('Score')
ax_bar.set_title('Model Comparison')
ax_bar.legend(loc='lower right')
ax_bar.axhline(y=1, linestyle='--', color='gray', alpha=0.5)

plt.suptitle('Machine Learning Results', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '07_ml_results.png', bbox_inches='tight')
plt.show()

---
## Section 8 — Results and Interpretation

In [ ]:
# Best model selection
best_model_name = df_results['F1-score'].idxmax()
best_model_obj  = {'SVM (RBF)': best_svm, f'KNN (k={best_k})': best_knn, 'Logistic Reg': lr}[best_model_name]
print(f"Best model: {best_model_name}")
print(df_results.loc[best_model_name].round(4))

In [ ]:
# Permutation feature importance
perm = permutation_importance(best_model_obj, X_test_s, y_test,
                               n_repeats=20, random_state=SEED, scoring='f1')

importance_mean = perm.importances_mean
importance_std  = perm.importances_std
sorted_idx = np.argsort(importance_mean)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    [FEATURE_NAMES[i] for i in sorted_idx],
    importance_mean[sorted_idx],
    xerr=importance_std[sorted_idx],
    color='steelblue', alpha=0.85, capsize=4
)
ax.set_xlabel('Mean decrease in F1-score')
ax.set_title(f'Permutation Feature Importance — {best_model_name}')
plt.tight_layout()
plt.savefig(FIGURES / '08_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics table per class
df_all = pd.DataFrame(X, columns=FEATURE_NAMES)
df_all['label'] = y

summary = df_all.groupby('label')[FEATURE_NAMES].mean().T
summary.columns = ['Good (mean)', 'Defective (mean)']
summary['Difference'] = summary['Defective (mean)'] - summary['Good (mean)']
print("\nFeature means per class:")
print(summary.round(4))
summary.to_csv(METRICS / 'class_statistics.csv')

In [ ]:
# Misclassified examples — show 6 with their full processing pipeline
best_preds = {'SVM (RBF)': y_pred_svm, f'KNN (k={best_k})': y_pred_knn, 'Logistic Reg': y_pred_lr}
y_pred_best = best_preds[best_model_name]

X_test_orig = X_test  # before scaling, for index mapping
# Get indices in the test split
test_paths_arr = np.array(paths)
_, test_indices = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=SEED, stratify=y
)
test_paths = [paths[i] for i in test_indices]

misclassified = [i for i, (pred, true) in enumerate(zip(y_pred_best, y_test)) if pred != true]
print(f"Misclassified: {len(misclassified)} / {len(y_test)}")

show_n = min(6, len(misclassified))
fig, axes = plt.subplots(show_n, 5, figsize=(18, show_n * 3))
if show_n == 1:
    axes = axes[np.newaxis, :]

stage_names_short = ['Equalised', 'Canny', 'Otsu', 'Closed', 'Original']

for row, idx in enumerate(misclassified[:show_n]):
    p = test_paths[idx]
    stages = preprocess(p)
    eq = stages['equalized']
    edges = canny(eq)
    mask  = segment_tire(eq)
    kernel = rect_kernel(3)
    closed = closing(mask, kernel)
    orig = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)

    imgs_row = [eq, edges, mask, closed, orig]
    cmaps_row = ['gray', 'gray', 'gray', 'gray', None]

    for col, (img, title, cmap) in enumerate(zip(imgs_row, stage_names_short, cmaps_row)):
        axes[row, col].imshow(img, cmap=cmap)
        if row == 0:
            axes[row, col].set_title(title)
        axes[row, col].axis('off')

    true_lbl  = 'Good' if y_test[idx] == 0 else 'Defective'
    pred_lbl  = 'Good' if y_pred_best[idx] == 0 else 'Defective'
    axes[row, 0].set_ylabel(f'True: {true_lbl}\nPred: {pred_lbl}',
                             fontsize=9, color='red', rotation=0, labelpad=80)

plt.suptitle(f'Misclassified Examples — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '08_misclassified.png', bbox_inches='tight')
plt.show()

In [ ]:
# Final summary print
print("=" * 55)
print(" FINAL RESULTS SUMMARY")
print("=" * 55)
print(df_results.round(4).to_string())
print(f"\nBest model : {best_model_name}")
print(f"Best F1    : {df_results.loc[best_model_name, 'F1-score']:.4f}")
print(f"Figures saved to  : {FIGURES.resolve()}")
print(f"Metrics saved to  : {METRICS.resolve()}")